# Hurricane Sandy: where did flood claims hit hardest, and did the flood maps see it coming?

The same three-step story as `01_sandy_spatial.sql`, told with maps and charts.

- **Step 1 · Types:** Manhattan ZIP codes as real shapes (GEOGRAPHY).
- **Step 2 · Functions:** counties near Manhattan, ranked by claim COUNT and by MEAN building payout.
- **Step 3 · Surprise:** were these properties even in the high-risk flood zone?

Data: FEMA NFIP claims plus ZIP and county boundaries from the free *Snowflake Public Data (Free)* listing. Claims are counted by the ZIP or county FEMA assigns; no individual claim is placed on a map.

In [ ]:
# Parameters. Change a knob, then Run all.
from snowflake.snowpark.context import get_active_session
session = get_active_session()
session.use_warehouse("SDSC_WORKSHOP_WH")

LISTING_DB = "SNOWFLAKE_PUBLIC_DATA_FREE"   # <-- change if you renamed it
CLAIMS_TBL = f"{LISTING_DB}.PUBLIC_DATA_FREE.FEMA_NATIONAL_FLOOD_INSURANCE_PROGRAM_CLAIM_INDEX"
GEO_CHARS  = f"{LISTING_DB}.PUBLIC_DATA_FREE.GEOGRAPHY_CHARACTERISTICS"
GEO_INDEX  = f"{LISTING_DB}.PUBLIC_DATA_FREE.GEOGRAPHY_INDEX"

WITHIN_M   = 6000    # Step 1: 'near the southern tip' radius, metres
DISTANCE_M = 25000   # Step 2: county search radius from Manhattan, metres (TRY: 75000)
MIN_CLAIMS = 500     # Step 2: minimum claims per county (TRY: 5000)

In [ ]:
import json
from decimal import Decimal
import pandas as pd
import altair as alt
alt.renderers.enable("mimetype")

BLUE, ORANGE = "#29B5E8", "#FF9F36"

def pdf(result):
    """SQL cell result -> pandas, with Decimal columns as floats so Altair can serialise them."""
    d = result.copy() if isinstance(result, pd.DataFrame) else result.to_pandas()
    for col in d.columns:
        if d[col].map(lambda v: isinstance(v, Decimal)).any():
            d[col] = d[col].astype(float)
    return d

def plain(v):
    return v.item() if hasattr(v, "item") else v

def features(df, props):
    """GeoJSON from ST_ASGEOJSON, used as-is (rings are never reversed)."""
    return alt.Data(values=[
        {"type": "Feature", "geometry": json.loads(r["GEOJSON"]),
         "properties": {k: plain(r[k]) for k in props}}
        for _, r in df.iterrows()])

def geomap(data, field, legend_title, scheme, fmt, tooltip, title, width=320, height=480):
    # Planar 'identity' projection: ring direction cannot turn a shape inside-out.
    return alt.Chart(data).mark_geoshape(stroke="white", strokeWidth=0.6).encode(
        color=alt.Color(f"properties.{field}:Q", title=legend_title,
                        scale=alt.Scale(scheme=scheme), legend=alt.Legend(format=fmt)),
        tooltip=tooltip,
    ).project(type="identity", reflectY=True).properties(width=width, height=height, title=title)

## Step 1 · Types: which Manhattan ZIP codes filed the most Sandy claims?

`TO_GEOGRAPHY` turns each ZIP boundary (WKT text) into a real shape. `ST_AREA`, `ST_CENTROID`, `ST_DISTANCE` and `ST_DWITHIN` then measure it on the Earth's surface, in metres. The query also counts claims outside FEMA's high-risk zone, which we use in Step 3.

In [ ]:
%%sql -r zips
WITH manhattan_claims AS (
  SELECT zip_geo_id,
         COUNT(*)                                                     AS claims,
         ROUND(AVG(amount_paid_on_building_claim))                    AS mean_bldg_paid,
         COUNT_IF(NOT policy_rated_flood_zone ILIKE 'Special Flood%') AS outside_high_risk_zone
  FROM {{CLAIMS_TBL}}
  WHERE flood_event = 'Hurricane Sandy'
    AND county_geo_id = 'geoId/36061'               -- Manhattan
  GROUP BY zip_geo_id
),
zip_shapes AS (
  SELECT geo_id, TO_GEOGRAPHY(value) AS geog
  FROM {{GEO_CHARS}}
  WHERE relationship_type = 'coordinates_wkt'
    AND geo_id IN (SELECT zip_geo_id FROM manhattan_claims)
),
southern_tip AS (
  -- ZIP 10004: Battery Park, where the surge came ashore.
  SELECT ST_CENTROID(geog) AS pt FROM zip_shapes WHERE geo_id = 'zip/10004'
)
SELECT REPLACE(c.zip_geo_id, 'zip/', '')                          AS zip,
       DECODE(zip, '10009','East Village', '10013','Tribeca / SoHo',
                   '10038','Seaport', '10011','Chelsea', '10029','East Harlem',
                   '10014','West Village', '10004','Battery Park',
                   '10128','Upper East Side', '10010','Gramercy',
                   '10035','East Harlem (north)', '10002','Lower East Side',
                   '10282','Battery Park City', '10280','Battery Park City',
                   '10003','East Village / Union Sq', 'Other')    AS neighborhood,
       c.claims,
       c.mean_bldg_paid,
       c.outside_high_risk_zone,
       ROUND(100 * c.outside_high_risk_zone / c.claims)            AS pct_outside,
       ROUND(ST_AREA(z.geog) / 1e6, 2)                             AS area_km2,
       ROUND(ST_DISTANCE(ST_CENTROID(z.geog), t.pt) / 1000, 1)     AS km_from_southern_tip,
       ST_DWITHIN(ST_CENTROID(z.geog), t.pt, {{WITHIN_M}})         AS near_southern_tip,
       ST_ASGEOJSON(z.geog)::STRING                                AS geojson
FROM manhattan_claims c
JOIN zip_shapes z ON z.geo_id = c.zip_geo_id
CROSS JOIN southern_tip t
ORDER BY c.claims DESC

### Map 1 · Manhattan ZIP codes coloured by Sandy claims

Darker blue = more claims. Hover for the neighbourhood, mean building payout, distance from the southern tip, and whether the ZIP is within `WITHIN_M` of it. **Notice:** the darkest ZIP (East Village, 10009) is not at the tip, and East Harlem, 12.5 km north, is still among the top five. The surge pushed up both rivers.

In [ ]:
z = pdf(zips)
zip_data = features(z, ["ZIP", "NEIGHBORHOOD", "CLAIMS", "MEAN_BLDG_PAID", "OUTSIDE_HIGH_RISK_ZONE",
                        "PCT_OUTSIDE", "KM_FROM_SOUTHERN_TIP", "NEAR_SOUTHERN_TIP"])
zip_tip = [alt.Tooltip("properties.ZIP:N", title="ZIP"),
           alt.Tooltip("properties.NEIGHBORHOOD:N", title="Neighbourhood"),
           alt.Tooltip("properties.CLAIMS:Q", title="Claims", format=","),
           alt.Tooltip("properties.MEAN_BLDG_PAID:Q", title="Mean building paid", format="$,.0f"),
           alt.Tooltip("properties.KM_FROM_SOUTHERN_TIP:Q", title="km from southern tip", format=".1f"),
           alt.Tooltip("properties.NEAR_SOUTHERN_TIP:N", title=f"Within {WITHIN_M/1000:g} km of tip"),
           alt.Tooltip("properties.PCT_OUTSIDE:Q", title="% outside high-risk zone", format=".0f")]

geomap(zip_data, "CLAIMS", "Sandy claims", "blues", ",.0f", zip_tip,
       "Manhattan: Hurricane Sandy flood claims by ZIP code")

## Step 2 · Functions: zoom out to counties and rank them two ways

`TRY_TO_GEOGRAPHY` turns county WKT into polygons (bad rows become NULL, not errors). `ST_DWITHIN` keeps counties whose boundary is within `DISTANCE_M` of Manhattan's, and `ST_SIMPLIFY(boundary, 200)` thins the outlines to 200 m so they draw fast. Mean paid excludes records with no payment; `PAID_MISSING` shows how many.

In [ ]:
%%sql -r counties
WITH county_claims AS (
  SELECT county_geo_id,
         COUNT(*)                                        AS claim_records,
         COUNT(*) - COUNT(amount_paid_on_building_claim) AS paid_missing,
         AVG(amount_paid_on_building_claim)              AS mean_bldg_paid_usd
  FROM {{CLAIMS_TBL}}
  WHERE flood_event   = 'Hurricane Sandy'
    AND state_geo_id IN ('geoId/36', 'geoId/34')    -- New York, New Jersey
    AND county_geo_id IS NOT NULL
  GROUP BY county_geo_id
  HAVING COUNT(*) >= {{MIN_CLAIMS}}
),
county_shapes AS (
  SELECT geo_id, TRY_TO_GEOGRAPHY(value) AS boundary
  FROM {{GEO_CHARS}}
  WHERE relationship_type = 'coordinates_wkt'
    AND geo_id IN (SELECT county_geo_id FROM county_claims UNION SELECT 'geoId/36061')
  QUALIFY ROW_NUMBER() OVER (PARTITION BY geo_id
                             ORDER BY relationship_start_date DESC NULLS LAST) = 1
),
manhattan AS (SELECT boundary FROM county_shapes WHERE geo_id = 'geoId/36061'),
nearby AS (
  SELECT c.*, s.boundary,
         ROUND(ST_DISTANCE(s.boundary, m.boundary) / 1000, 1) AS km_from_manhattan
  FROM county_claims c
  JOIN county_shapes s ON s.geo_id = c.county_geo_id
  CROSS JOIN manhattan m
  WHERE ST_DWITHIN(s.boundary, m.boundary, {{DISTANCE_M}})
)
SELECT g.geo_name                                       AS county,
       claim_records,
       paid_missing,
       ROUND(mean_bldg_paid_usd)                        AS mean_bldg_paid_usd,
       km_from_manhattan,
       RANK() OVER (ORDER BY claim_records DESC)        AS rank_by_count,
       RANK() OVER (ORDER BY mean_bldg_paid_usd DESC)   AS rank_by_mean_paid,
       ST_ASGEOJSON(ST_SIMPLIFY(boundary, 200))::STRING AS geojson
FROM nearby n
JOIN {{GEO_INDEX}} g ON g.geo_id = n.county_geo_id
ORDER BY rank_by_count

### Map 2 · The same counties, coloured two ways

Left: claim count (blues). Right: mean building payout (oranges). Each map has its own colour scale. **Notice:** Nassau dominates on the left, but on the right Manhattan is the darkest county. Fewer claims, much bigger buildings.

In [ ]:
c = pdf(counties)
county_data = features(c, ["COUNTY", "CLAIM_RECORDS", "MEAN_BLDG_PAID_USD", "PAID_MISSING",
                           "KM_FROM_MANHATTAN", "RANK_BY_COUNT", "RANK_BY_MEAN_PAID"])
county_tip = [alt.Tooltip("properties.COUNTY:N", title="County"),
              alt.Tooltip("properties.CLAIM_RECORDS:Q", title="Claims", format=","),
              alt.Tooltip("properties.MEAN_BLDG_PAID_USD:Q", title="Mean building paid", format="$,.0f"),
              alt.Tooltip("properties.PAID_MISSING:Q", title="No payment recorded", format=","),
              alt.Tooltip("properties.KM_FROM_MANHATTAN:Q", title="km from Manhattan", format=".1f"),
              alt.Tooltip("properties.RANK_BY_COUNT:Q", title="Rank by count"),
              alt.Tooltip("properties.RANK_BY_MEAN_PAID:Q", title="Rank by mean paid")]

alt.hconcat(
    geomap(county_data, "CLAIM_RECORDS", "Claims", "blues", ",.0f", county_tip,
           "By claim count", width=380, height=420),
    geomap(county_data, "MEAN_BLDG_PAID_USD", "Mean paid", "oranges", "$,.0f", county_tip,
           "By mean building paid", width=380, height=420),
    title=f"Counties within {DISTANCE_M/1000:g} km of Manhattan with at least {MIN_CLAIMS:,} Sandy claims",
).resolve_scale(color="independent")

### Chart 3 · Slope chart: rank by count vs rank by mean paid

Each line is one county moving from its rank by claim count (left) to its rank by mean building payout (right); 1 is the top. Manhattan is highlighted in orange. **Notice:** Manhattan jumps from #9 to #1. The metric you pick decides the answer.

In [ ]:
ranks = c.melt(id_vars=["COUNTY"], value_vars=["RANK_BY_COUNT", "RANK_BY_MEAN_PAID"],
               var_name="RANKING", value_name="RANK")
ranks["RANKING"] = ranks["RANKING"].map({"RANK_BY_COUNT": "By claim count",
                                         "RANK_BY_MEAN_PAID": "By mean paid"})
highlight = alt.condition(alt.datum.COUNTY == "Manhattan", alt.value(ORANGE), alt.value(BLUE))
slope = alt.Chart(ranks).encode(
    x=alt.X("RANKING:N", title=None, sort=["By claim count", "By mean paid"],
            axis=alt.Axis(labelAngle=0, orient="top")),
    y=alt.Y("RANK:Q", scale=alt.Scale(reverse=True, domain=[1, len(c)]), title="Rank (1 = top)"),
    detail="COUNTY:N",
    color=highlight,
    tooltip=["COUNTY", "RANKING", "RANK"],
)
(slope.mark_line(point=True, strokeWidth=2)
 + slope.mark_text(align="right", dx=-8).encode(text="COUNTY:N")
        .transform_filter(alt.datum.RANKING == "By claim count")
 + slope.mark_text(align="left", dx=8).encode(text="COUNTY:N")
        .transform_filter(alt.datum.RANKING == "By mean paid")
).properties(width=420, height=400, title="The story flips: rank by claim count vs rank by mean building paid")

### Chart 4 · Drag the threshold: re-rank by mean paid in the browser

The SQL below fetches every NY/NJ county with at least 100 Sandy claims once. The slider then filters by minimum claims and re-ranks the survivors by mean building payout entirely in the browser (an Altair window `rank()`), with no SQL rerun. **Try:** drag to 5,000 and watch Manhattan disappear.

In [ ]:
%%sql -r all_counties
SELECT g.geo_name                                   AS county,
       COUNT(*)                                     AS claim_records,
       ROUND(AVG(c.amount_paid_on_building_claim))  AS mean_bldg_paid_usd
FROM {{CLAIMS_TBL}} c
JOIN {{GEO_INDEX}} g ON g.geo_id = c.county_geo_id
WHERE c.flood_event = 'Hurricane Sandy'
  AND c.state_geo_id IN ('geoId/36', 'geoId/34')
GROUP BY c.county_geo_id, g.geo_name
HAVING COUNT(*) >= 100

In [ ]:
ac = pdf(all_counties)
threshold = alt.param(name="min_claims", value=MIN_CLAIMS,
                      bind=alt.binding_range(min=100, max=10000, step=100, name="Minimum claims "))
ranked = alt.Chart(ac).transform_filter(
    alt.datum.CLAIM_RECORDS >= threshold
).transform_window(
    RANK="rank()", sort=[alt.SortField("MEAN_BLDG_PAID_USD", order="descending")]
).transform_calculate(
    LABEL="'#' + datum.RANK"
).encode(
    y=alt.Y("COUNTY:N", sort=alt.EncodingSortField("MEAN_BLDG_PAID_USD", order="descending"), title=None),
    x=alt.X("MEAN_BLDG_PAID_USD:Q", title="Mean building paid (USD)", axis=alt.Axis(format="$,.0f")),
    tooltip=[alt.Tooltip("COUNTY:N", title="County"),
             alt.Tooltip("RANK:Q", title="Rank by mean paid"),
             alt.Tooltip("CLAIM_RECORDS:Q", title="Claims", format=","),
             alt.Tooltip("MEAN_BLDG_PAID_USD:Q", title="Mean building paid", format="$,.0f")],
)
alt.layer(
    ranked.mark_bar(color=ORANGE),
    ranked.mark_text(align="left", dx=4, fontWeight="bold").encode(text="LABEL:N"),
).add_params(threshold).properties(
    width=460, title="NY/NJ counties above the claims threshold, ranked by mean building paid")

### Chart 5 · Does the distance we pick change the answer?

A Python loop reruns the Step 2 spatial query at 0, 10, ... 80 km from Manhattan, binding the distance as a query parameter (with `MIN_CLAIMS` applied). The line shows total claims captured; each point is coloured by the #1 county by claims at that distance. **Notice:** the #1 county changes once Ocean County, NJ (the biggest of the event) comes into range.

In [ ]:
sweep_sql = f"""
WITH county_claims AS (
  SELECT county_geo_id, COUNT(*) AS claim_records
  FROM {CLAIMS_TBL}
  WHERE flood_event = 'Hurricane Sandy'
    AND state_geo_id IN ('geoId/36', 'geoId/34')
    AND county_geo_id IS NOT NULL
  GROUP BY county_geo_id
  HAVING COUNT(*) >= ?
),
county_shapes AS (
  SELECT geo_id, TRY_TO_GEOGRAPHY(value) AS boundary
  FROM {GEO_CHARS}
  WHERE relationship_type = 'coordinates_wkt'
    AND geo_id IN (SELECT county_geo_id FROM county_claims UNION SELECT 'geoId/36061')
  QUALIFY ROW_NUMBER() OVER (PARTITION BY geo_id
                             ORDER BY relationship_start_date DESC NULLS LAST) = 1
)
SELECT g.geo_name AS county, c.claim_records
FROM county_claims c
JOIN county_shapes s ON s.geo_id = c.county_geo_id
JOIN {GEO_INDEX} g ON g.geo_id = c.county_geo_id
CROSS JOIN (SELECT boundary FROM county_shapes WHERE geo_id = 'geoId/36061') m
WHERE ST_DWITHIN(s.boundary, m.boundary, ?)
"""

rows = []
for km in range(0, 81, 10):
    found = pd.DataFrame(session.sql(sweep_sql, params=[MIN_CLAIMS, km * 1000]).collect())
    top = found.sort_values("CLAIM_RECORDS", ascending=False).iloc[0]
    rows.append({"KM": km, "COUNTIES": len(found), "CLAIMS": int(found["CLAIM_RECORDS"].sum()),
                 "TOP_COUNTY": top["COUNTY"], "TOP_CLAIMS": int(top["CLAIM_RECORDS"])})
sweep = pd.DataFrame(rows)

line = alt.Chart(sweep).encode(
    x=alt.X("KM:Q", title="Distance from Manhattan (km)", scale=alt.Scale(domain=[0, 80])),
    y=alt.Y("CLAIMS:Q", title="Sandy claims captured", axis=alt.Axis(format=",")),
    tooltip=[alt.Tooltip("KM:Q", title="km"), alt.Tooltip("COUNTIES:Q", title="Counties"),
             alt.Tooltip("CLAIMS:Q", title="Claims captured", format=","),
             alt.Tooltip("TOP_COUNTY:N", title="#1 county"),
             alt.Tooltip("TOP_CLAIMS:Q", title="#1 county claims", format=",")],
)
(line.mark_line(color=BLUE, strokeWidth=2)
 + line.mark_point(filled=True, size=120, opacity=1).encode(
     color=alt.Color("TOP_COUNTY:N", title="#1 county by claims",
                     scale=alt.Scale(range=[BLUE, ORANGE, "#11567F", "#FFC27A"])))
).properties(width=560, height=320, title="The circle you draw decides the answer: claims captured by distance from Manhattan")

## Step 3 · The surprise: were these properties even in the flood zone?

`POLICY_RATED_FLOOD_ZONE` is the FEMA zone each policy was rated in at the time of the loss; "Special Flood..." is FEMA's high-risk zone (A and V). Left: the top 10 Manhattan ZIPs by claims, split into claims inside (blue) and outside (orange) the high-risk zone. Right: the share of each ZIP's claims that came from outside it. **Notice:** waterfront Tribeca and Seaport were mostly mapped correctly, but a third of East Village claims and half of East Harlem's came from outside the zone.

In [ ]:
top10 = z.head(10).assign(INSIDE=lambda d: d.CLAIMS - d.OUTSIDE_HIGH_RISK_ZONE)
zone = top10.melt(id_vars=["ZIP", "NEIGHBORHOOD", "PCT_OUTSIDE"],
                  value_vars=["INSIDE", "OUTSIDE_HIGH_RISK_ZONE"], var_name="ZONE", value_name="N")
zone["ZONE"] = zone["ZONE"].map({"INSIDE": "Inside high-risk zone",
                                 "OUTSIDE_HIGH_RISK_ZONE": "Outside high-risk zone"})
bars = alt.Chart(zone).mark_bar().encode(
    x=alt.X("N:Q", title="Sandy claims", stack="zero"),
    y=alt.Y("ZIP:N", sort=list(top10.ZIP), title=None),
    color=alt.Color("ZONE:N", title=None, scale=alt.Scale(range=[BLUE, ORANGE]),
                    legend=alt.Legend(orient="bottom")),
    order=alt.Order("ZONE:N"),
    tooltip=[alt.Tooltip("ZIP:N"), alt.Tooltip("NEIGHBORHOOD:N", title="Neighbourhood"),
             alt.Tooltip("ZONE:N", title="Zone"), alt.Tooltip("N:Q", title="Claims", format=","),
             alt.Tooltip("PCT_OUTSIDE:Q", title="% outside", format=".0f")],
).properties(width=360, height=320, title="Top 10 Manhattan ZIPs: claims inside vs outside the high-risk zone")

outside_map = geomap(zip_data, "PCT_OUTSIDE", "% outside", "oranges", ".0f", zip_tip,
                     "Share of claims from outside the high-risk zone")

(bars | outside_map).resolve_scale(color="independent")

### Checks

The cell below confirms the headline numbers and that every shape is a sensible size (a broken, inside-out shape would cover the globe).

In [ ]:
manhattan = c.set_index("COUNTY").loc["Manhattan"]
e_village = z.set_index("ZIP").loc["10009"]
checks = {
    "Top ZIP is 10009 with 94 claims": z.iloc[0]["ZIP"] == "10009" and int(z.iloc[0]["CLAIMS"]) == 94,
    "Manhattan #9 by count (25 km)": int(manhattan["RANK_BY_COUNT"]) == 9,
    "Manhattan #1 by mean paid ($236,483)": int(manhattan["RANK_BY_MEAN_PAID"]) == 1
        and int(manhattan["MEAN_BLDG_PAID_USD"]) == 236483,
    "10009: 31 of 94 outside zone (33%)": int(e_village["OUTSIDE_HIGH_RISK_ZONE"]) == 31
        and int(e_village["PCT_OUTSIDE"]) == 33,
    "Every ZIP shape is under 20 km2": bool((z["AREA_KM2"] < 20).all()),
    "Distinct county outlines (one per county)": c["GEOJSON"].nunique() == len(c),
}
for name, ok in checks.items():
    print(f"{'PASS' if ok else 'FAIL'}  {name}")

### What you just did

You turned WKT text into GEOGRAPHY, measured areas and distances in metres, filtered by distance with `ST_DWITHIN`, simplified boundaries for fast maps, and saw how the area you draw and the metric you pick change the answer. Change a knob in the Parameters cell and Run all to see the story move. This is descriptive history for one event, not a flood risk model.